#  Fraud Detection Pipeline
### Project 2 — Supervised Learning | DecodeLabs Industrial Training Kit | Batch 2026

---

##  Project Overview

This notebook builds a **complete fraud detection pipeline** on an e-commerce orders dataset.  
The goal is to classify transactions as **Fraudulent (1)** or **Legitimate (0)** using supervised machine learning.

### Key Requirements from the Task:
- Handle class imbalance using **SMOTE** (Synthetic Minority Over-sampling Technique)
- Train two classifiers: **Logistic Regression** and **Random Forest**
- Use **imblearn Pipeline** to prevent data leakage
- Evaluate using **Precision, Recall, ROC-AUC** — Accuracy is intentionally discarded
- Tune hyperparameters using **GridSearchCV**

---

## Table of Contents

1. [Dataset Preparation & Feature Engineering](#step1)
2. [Stratified Train/Test Split](#step2)
3. [Build imblearn Pipelines](#step3)
4. [Hyperparameter Tuning — GridSearchCV](#step4)
5. [Model Evaluation](#step5)
6. [Visualizations](#step6)
7. [Conclusion](#conclusion)


---
<a id='step1'></a>
##  Step 1 — Dataset Preparation & Feature Engineering

### What we do in this step:
- Load the pre-cleaned dataset
- Create the **IsFraud** target column from `OrderStatus`
- Extract time features (`Month`, `DayOfWeek`) from `Date`
- Encode categorical columns into numbers
- Drop useless columns (IDs, zero-variance, leakage columns)
- Separate features `X` and target `y`
- Save the final ML-ready dataset

### Why this matters:
Machine learning models only understand **numbers**. Raw text columns like `PaymentMethod = 'Credit Card'` or `Product = 'Laptop'` must be converted. Also, columns like `OrderID` are unique per row and teach the model nothing — so they are dropped.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#### Load Dataset

In [ ]:
df = pd.read_csv('cleaned_dataset.xls')

print("Dataset Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

####  Create Fraud Label — IsFraud Column

**Business Logic:**
- `Cancelled` and `Returned` orders → **1 (Fraud)** — suspicious behavior in e-commerce
- `Shipped`, `Delivered`, `Pending` → **0 (Legitimate)** — normal order flow

`.isin()` checks each row against the list → returns `True/False` → `.astype(int)` converts to `1/0`


In [ ]:
df["IsFraud"] = df["OrderStatus"].isin(["Cancelled", "Returned"]).astype(int)

print("IsFraud Distribution:")
print(df["IsFraud"].value_counts())
print(f"\nFraud Rate: {df['IsFraud'].mean()*100:.1f}%")

####  Extract Date Features

Instead of using the raw `Date` column (which has no direct pattern), we extract:
- `Month` → fraud may spike in certain months (e.g. holiday season)
- `DayOfWeek` → fraud may be higher on weekends (0=Monday, 6=Sunday)


In [ ]:
df["Date"]      = pd.to_datetime(df["Date"])
df["Month"]     = df["Date"].dt.month
df["DayOfWeek"] = df["Date"].dt.dayofweek

print(df[["Date", "Month", "DayOfWeek"]].head(3))

####  Encode Categorical Columns

Models cannot read text. We manually map each category to a number:
- `PaymentMethod` → Cash=0, Debit Card=1, Credit Card=2, Gift Card=3, Online=4
- `Product` → Chair=0, Desk=1, Laptop=2, Monitor=3, Phone=4, Printer=5, Tablet=6
- `ReferralSource` → Email=0, Facebook=1, Google=2, Instagram=3, Referral=4


In [ ]:
pm_map   = {'Cash':0, 'Debit Card':1, 'Credit Card':2, 'Gift Card':3, 'Online':4}
prod_map = {'Chair':0, 'Desk':1, 'Laptop':2, 'Monitor':3, 'Phone':4, 'Printer':5, 'Tablet':6}
ref_map  = {'Email':0, 'Facebook':1, 'Google':2, 'Instagram':3, 'Referral':4}

df['PaymentMethod_enc']  = df['PaymentMethod'].map(pm_map)
df['Product_enc']        = df['Product'].map(prod_map)
df['ReferralSource_enc'] = df['ReferralSource'].map(ref_map)

print(df[['PaymentMethod', 'PaymentMethod_enc']].head(3))

####  Drop Useless Columns

| Column | Reason to Drop |
|---|---|
| `OrderID`, `CustomerID`, `TrackingNumber` | Unique IDs — no repeating pattern |
| `ShippingAddress` | High cardinality text — not encodable |
| `CouponCode` | Random codes, no pattern |
| `HasCoupon` | Zero variance — all rows = 1 |
| `OrderStatus` | Source of IsFraud — would cause data leakage |
| `Date` | Already extracted Month & DayOfWeek from it |
| `PaymentMethod`, `Product`, `ReferralSource` | Replaced by their encoded versions |


In [ ]:
cols_to_drop = ['OrderID', 'CustomerID', 'TrackingNumber',
                'ShippingAddress', 'CouponCode', 'HasCoupon',
                'OrderStatus', 'Date',
                'PaymentMethod', 'Product', 'ReferralSource']

df.drop(columns=cols_to_drop, inplace=True)

print("Remaining columns:", df.columns.tolist())
print("Total columns:", len(df.columns))

####  Separate Features (X) and Target (y) — Final Verification

In [ ]:
X = df.drop(columns=['IsFraud'])   # 11 feature columns
y = df['IsFraud']                  # target column

print("X shape:", X.shape)        # should be (1200, 11)
print("y shape:", y.shape)        # should be (1200,)
print("\nMissing values:", X.isnull().sum().sum())   # should be 0
print("\nFeature columns:\n", X.columns.tolist())
print("\nClass balance:")
print(y.value_counts())

####  Save ML-Ready Dataset

In [ ]:
ml_df = X.copy()
ml_df['IsFraud'] = y

ml_df.to_csv('ml_ready_dataset.csv', index=False)
print("Saved! Shape:", ml_df.shape)
ml_df.head()

** Step 1 Complete**
- Dataset shape: **1200 rows × 11 features + 1 target**
- Missing values: **0**
- Fraud (1): **497 rows** | Legitimate (0): **703 rows**
- All columns are now **numeric** — ready for ML


---
<a id='step2'></a>
##  Step 2 — Stratified Train/Test Split (80/20)

### What we do:
Split the dataset into 80% training and 20% test sets.

### Why stratified?
`stratify=y` ensures **both sets maintain the same fraud ratio (~41%)**.  
Without it, all fraud cases could accidentally land in one set, making evaluation unreliable.

>  **Golden Rule:** The test set must **never** be touched by SMOTE or any preprocessing — it must reflect real-world unseen data.


In [ ]:
from sklearn.model_selection import train_test_split

X = ml_df.drop(columns=['IsFraud'])
y = ml_df['IsFraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% for testing
    random_state=42,    # reproducible results every run
    stratify=y          # preserve fraud % in both sets
)

print("X_train shape:", X_train.shape)   # 80% → 960 rows
print("X_test shape: ", X_test.shape)    # 20% → 240 rows
print("Train fraud %:", round(y_train.mean()*100, 2))
print("Test  fraud %:", round(y_test.mean()*100, 2))

** Step 2 Complete**
- Training set: **960 rows** | Test set: **240 rows**
- Both sets have the **same fraud %** — stratification worked correctly ✓


---
<a id='step3'></a>
##  Step 3 — Build imblearn Pipelines

### What is SMOTE?
**S**ynthetic **M**inority **O**ver-sampling **TE**chnique — creates new **synthetic** fraud examples by interpolating between existing real fraud rows. It does NOT copy rows (which causes overfitting) — it *creates* new realistic ones.

### Why imblearn Pipeline and not sklearn Pipeline?
`sklearn.pipeline.Pipeline` **ignores** resampling steps.  
`imblearn.pipeline.Pipeline` correctly applies SMOTE **only on the training fold** during cross-validation — never on the validation/test fold. This prevents data leakage.

### Pipeline Design:
| Pipeline | Steps | Reason |
|---|---|---|
| **A — Logistic Regression** | StandardScaler → SMOTE → LR | LR needs scaling — large features distort regularization |
| **B — Random Forest** | SMOTE → RF | RF is scale-invariant — tree splits are unaffected by magnitude |


In [ ]:
# Install imbalanced-learn if not already installed
!pip install imbalanced-learn --quiet

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Pipeline A — Logistic Regression (needs StandardScaler)
lr_pipeline = ImbPipeline([
    ('scaler',     StandardScaler()),
    ('smote',      SMOTE(random_state=42)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Pipeline B — Random Forest (scale-invariant, no scaler needed)
rf_pipeline = ImbPipeline([
    ('smote',      SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42))
])

print("Pipeline A (LR):", [step[0] for step in lr_pipeline.steps])
print("Pipeline B (RF):", [step[0] for step in rf_pipeline.steps])

** Step 3 Complete**
- Both pipelines built using `imblearn.pipeline.Pipeline`
- SMOTE is **inside** the pipeline — applied only on training folds ✓
- Test data will **never** be seen by SMOTE ✓


---
<a id='step4'></a>
##  Step 4 — Hyperparameter Tuning with GridSearchCV

### What is GridSearchCV?
It automatically tries **every combination** of hyperparameters and picks the best one.  
Combined with `StratifiedKFold`, it runs 5-fold cross-validation for every combination — keeping SMOTE safely inside each training fold.

### Parameters we tune:
| Parameter | Values Tried | What it controls |
|---|---|---|
| `smote__k_neighbors` | 3, 5 | How many neighbors SMOTE uses to generate synthetic samples |
| `classifier__C` (LR) | 0.01, 0.1, 1.0 | Regularization strength — lower = stronger regularization |
| `classifier__n_estimators` (RF) | 100, 200 | Number of trees in the forest |
| `classifier__max_depth` (RF) | 10, 20, None | Maximum depth of each tree |

> We use `scoring='roc_auc'` — **not accuracy** — so the tuning optimizes for the right metric.


In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# 5-fold stratified cross validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Hyperparameter grid for Logistic Regression
lr_params = {
    'smote__k_neighbors' : [3, 5],
    'classifier__C'      : [0.01, 0.1, 1.0]
}

print("Tuning Logistic Regression...")
lr_grid = GridSearchCV(lr_pipeline, lr_params,
                       cv=cv, scoring='roc_auc')
lr_grid.fit(X_train, y_train)

print("Best Parameters:", lr_grid.best_params_)
print("Best CV ROC-AUC:", round(lr_grid.best_score_, 4))

In [ ]:
# Hyperparameter grid for Random Forest
rf_params = {
    'smote__k_neighbors'      : [3, 5],
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth'   : [10, 20, None]
}

print("Tuning Random Forest (takes ~1 minute)...")
rf_grid = GridSearchCV(rf_pipeline, rf_params,
                       cv=cv, scoring='roc_auc')
rf_grid.fit(X_train, y_train)

print("Best Parameters:", rf_grid.best_params_)
print("Best CV ROC-AUC:", round(rf_grid.best_score_, 4))

** Step 4 Complete**
- Best LR config: `C=0.1, k_neighbors=5` → CV ROC-AUC = **0.526**
- Best RF config: `max_depth=20, n_estimators=200, k_neighbors=5` → CV ROC-AUC = **0.515**
- GridSearchCV ensured SMOTE was applied **inside every fold** — zero leakage ✓


---
<a id='step5'></a>
##  Step 5 — Model Evaluation

### Why we discard Accuracy:
In imbalanced classification, a model that predicts **"Legitimate" for every transaction** gets 58% accuracy — while catching **zero fraud**. Accuracy is completely misleading here.

### The metrics we use instead:

| Metric | Formula | What it answers |
|---|---|---|
| **Precision** | TP / (TP + FP) | When we flag fraud — how often are we right? |
| **Recall** | TP / (TP + FN) | Of all real fraud — how much did we catch? |
| **F1-Score** | 2 × (P × R) / (P + R) | Balance between Precision and Recall |
| **ROC-AUC** | Area under ROC curve | Overall ability to separate fraud from legitimate |

> Both models are evaluated on the **untouched 240-row test set** — data they have never seen before.


In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, precision_score,
                              recall_score, f1_score)

best_lr = lr_grid.best_estimator_
best_rf = rf_grid.best_estimator_

for name, model in [('Logistic Regression', best_lr),
                    ('Random Forest',       best_rf)]:

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print("=" * 50)
    print(f"  {name}")
    print("=" * 50)
    print(f"Precision : {precision_score(y_test, y_pred):.4f}")
    print(f"Recall    : {recall_score(y_test, y_pred):.4f}")
    print(f"F1-Score  : {f1_score(y_test, y_pred):.4f}")
    print(f"ROC-AUC   : {roc_auc_score(y_test, y_prob):.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred,
          target_names=['Legitimate', 'Fraud']))
    cm = confusion_matrix(y_test, y_pred)
    print(f"Confusion Matrix:")
    print(f"  TN={cm[0,0]}  FP={cm[0,1]}")
    print(f"  FN={cm[1,0]}  TP={cm[1,1]}\n")

** Step 5 Complete — Results Summary:**

| Metric | Logistic Regression | Random Forest |
|---|---|---|
| Precision | 0.367 | 0.378 |
| Recall | 0.404 | 0.343 |
| F1-Score | 0.385 | 0.360 |
| **ROC-AUC** | 0.463 | **0.469** ✓ |

** Winner: Random Forest** — higher ROC-AUC score

> **Note on scores:** ROC-AUC near 0.50 indicates the features in this e-commerce dataset do not strongly distinguish fraudulent from legitimate orders. This is a **data finding**, not a pipeline error — the pipeline architecture is correctly implemented per industry standards.


---
<a id='step6'></a>
##  Step 6 — Visualizations

### Chart 1 — ROC Curve:
Shows how well each model separates fraud from legitimate across all classification thresholds.  
A curve closer to the **top-left corner** = better model. The diagonal line = random guessing (AUC = 0.50).

### Chart 2 — Feature Importance (Random Forest):
Shows which features the Random Forest found most useful for detecting fraud.  
Higher bar = more important feature for the prediction.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── ROC Curve ───────────────────────────────────────────
for name, model, color in [('Logistic Regression', best_lr, '#3B82F6'),
                            ('Random Forest',       best_rf, '#EF4444')]:
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, lw=2)

axes[0].plot([0,1], [0,1], '--', color='gray', label='Random Classifier')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve Comparison')
axes[0].legend()
axes[0].grid(alpha=0.3)

# ── Feature Importance ──────────────────────────────────
rf_clf      = best_rf.named_steps['classifier']
importances = rf_clf.feature_importances_
feat_names  = X.columns
sorted_idx  = importances.argsort()

axes[1].barh([feat_names[i] for i in sorted_idx],
             importances[sorted_idx], color='#EF4444')
axes[1].set_title('Random Forest — Feature Importance')
axes[1].set_xlabel('Importance Score')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('fraud_detection_report.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved as fraud_detection_report.png")

**Step 6 Complete — Chart Observations:**
- **ROC Curve:** Both models track close to the diagonal, consistent with AUC ~0.46-0.47
- **Feature Importance:** `TotalPrice`, `UnitPrice`, and `RevenuePerItem` are the top 3 features — price-related signals carry the most (if still weak) fraud information
- `Quantity` ranked last — order size alone is not a strong fraud indicator


---
<a id='conclusion'></a>
##  Conclusion

This notebook successfully implemented a **complete, leak-free fraud detection pipeline** satisfying all project requirements:

| Requirement | Status |
|---|---|
| SMOTE for class imbalance handling | ✅ Applied inside imblearn Pipeline |
| Logistic Regression model | ✅ Tuned with GridSearchCV |
| Random Forest model | ✅ Tuned with GridSearchCV |
| No data leakage | ✅ SMOTE inside CV folds only |
| Accuracy discarded | ✅ Evaluated with Precision, Recall, ROC-AUC |
| Hyperparameter tuning | ✅ GridSearchCV with 5-Fold Stratified CV |

**Best Model: Random Forest** (ROC-AUC = 0.469)

**Key Insight:** The modest ROC-AUC scores (~0.47) reflect the nature of the dataset — this is a general e-commerce order log, not a purpose-built fraud dataset. The proxy label (Cancelled/Returned = Fraud) captures normal customer behavior as much as actual fraud. In a real financial system, dedicated fraud signals (IP mismatch, transaction velocity, device fingerprinting) would yield significantly higher AUC scores.

---
*DecodeLabs Industrial Training Kit — Project 2 | Batch 2026*
